In [0]:
import pyspark.sql.functions as F
import pandas
from datetime import datetime, timedelta
from pyspark.sql.window import Window
import os
from marel.services import ServiceProvider
from marel.databricks import DatabricksProvider
from pyspark.ml.feature import StandardScaler
from pyspark.ml.functions import vector_to_array

from pyspark.ml.classification import MultilayerPerceptronClassifier
import mlflow

provider = DatabricksProvider()
table_service = provider.table_service

In [0]:

df_csv = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("multiLine", True) \
    .option("escape", "\"") \
    .option("quote", "\"") \
    .option("delimiter", ",") \
    .load("/Volumes/teams/sensorx/data-dump/deviceID_Serial_GenSerial_machineCode.csv")

# Device IDs that have at least one 800W serial number
devices_800w = (
    df_csv.filter(F.col("gentype") == "800W")
    .select("properties_deviceId")
    .distinct()
)

In [0]:
from pyspark.ml.feature import VectorAssembler

def df_read_data(days):

    # Read telemetry data
    df_telemetry = table_service.load_table_from_device_checkpoint(
        table_name = "prod_medallion.silver_machine.sensorx_telemetry",
        machine_name="sensorx",
        max_historical_days=days,
        old_format=True
    ).select("properties_deviceID",
            "properties_payloadTimeStamp",
            "payload_xrayController_filamentCurrent",
            "payload_xrayController_onTime",
            "payload_xrayController_temperature",
            "payload_xrayController_tubeCurrent",
            "payload_xrayController_tubeVoltage"
            )

    # Filter to 800w devices
    df_telemetry_800w = df_telemetry.join(
        F.broadcast(devices_800w), df_telemetry["properties_deviceID"] == devices_800w["properties_deviceId"], "inner"
    ).drop(devices_800w["properties_deviceId"])

    # Reading fault tables
    df_fault = (
        table_service.load_table_from_device_checkpoint(
            table_name = "prod_medallion.silver_machine.pluto_xraycontroller_fault_property",
            machine_name="sensorx",
            max_historical_days=days,
            old_format=True
        )
        .filter(F.col("payload_fault_faultName") == "faultRegulation")
        .select("properties_payloadTimeStamp",
                "properties_deviceID",
                "payload_fault_state")
    )

    # --- Union + forward-fill ---
    telem_only = [c for c in df_telemetry_800w.columns
                if c not in ("properties_payloadTimeStamp", "properties_deviceID")]

    telem_part = df_telemetry_800w.select(
        "properties_payloadTimeStamp", "properties_deviceID",
        *telem_only,
        F.lit(None).cast("boolean").alias("payload_fault_state"),
        F.lit(True).alias("_is_telem"),
    )

    fault_part = df_fault.select(
        "properties_payloadTimeStamp", "properties_deviceID",
        *[F.lit(None).cast(dict(df_telemetry_800w.dtypes)[c]).alias(c)
        for c in telem_only],
        "payload_fault_state",
        F.lit(False).alias("_is_telem"),
    )

    w = Window.partitionBy("properties_deviceID").orderBy("properties_payloadTimeStamp")

    df = (
        telem_part.unionByName(fault_part)
        .withColumn("payload_fault_state",
                    F.last("payload_fault_state", ignorenulls=True).over(w))
        .filter(F.col("_is_telem"))
        .drop("_is_telem")
    )

    df = df.withColumn(
        "payload_fault_state", F.coalesce(F.col("payload_fault_state"), F.lit(False))
    )

    # Adding delta seconds feature
    df = (
        df
        .withColumn("prev_ts", F.lag("properties_payloadTimeStamp").over(w))
        .withColumn(
            "delta_seconds",
            F.when(
                F.col("prev_ts").isNull() |
                (F.col("properties_payloadTimeStamp").cast("long") - F.col("prev_ts").cast("long") < 0),
                None
            ).otherwise(
                F.col("properties_payloadTimeStamp").cast("long") - F.col("prev_ts").cast("long")
            ).cast("double")
        )
        .drop("prev_ts")
    )

    # Feature engineering
    sensor_cols = [c for c in df.columns
                   if c not in {"properties_deviceID", "properties_payloadTimeStamp", "payload_fault_state"}]

    df = df.na.drop(subset=sensor_cols)

    # --- Device ID index: use training data's mapping ---
    # Known devices get their trained index; unknown devices get the mean index
    # (after normalization, mean index → 0 → truly neutral, model predicts from sensor patterns only)
    train_meta = spark.read.table("teams.sensorx.train_data").schema["features"].metadata
    device_vals = train_meta["ml_attr"]["attrs"]["nominal"][0]["vals"]
    default_index = float(len(device_vals) - 1) / 2.0  # mean of indices 0..N-1
    device_mapping = spark.createDataFrame(
        [(dev_id, float(idx)) for idx, dev_id in enumerate(device_vals)],
        ["properties_deviceID", "deviceId_index"]
    )
    df = df.join(F.broadcast(device_mapping), on="properties_deviceID", how="left")
    df = df.withColumn("deviceId_index", F.coalesce(F.col("deviceId_index"), F.lit(default_index)))

    # Lag features
    n_lags = 20
    w_lag = Window.partitionBy("properties_deviceID").orderBy("properties_payloadTimeStamp")

    lag_exprs = [F.lag(col_name, L).over(w_lag).alias(f"{col_name}{L}")
                 for L in range(1, n_lags + 1) for col_name in sensor_cols]
    df = df.select("*", *lag_exprs)

    lag_cols = [f"{col}{L}" for L in range(1, n_lags + 1) for col in sensor_cols]

    df = df.na.drop(subset=lag_cols)

    # --- Deviation features: 24-hour time-based rolling average ---
    w_daily = (
        Window.partitionBy("properties_deviceID")
        .orderBy(F.col("properties_payloadTimeStamp").cast("long"))
        .rangeBetween(-86400, 0)
    )

    dev_sensors = [
        "payload_xrayController_filamentCurrent",
        "payload_xrayController_temperature",
        "payload_xrayController_tubeCurrent",
    ]
    dev_cols = []
    dev_exprs = []

    for col_name in dev_sensors:
        avg_col = f"{col_name}_avg_daily"
        dev_col = f"{col_name}_dev_daily"
        dev_cols.extend([avg_col, dev_col])
        dev_exprs.append(F.avg(col_name).over(w_daily).alias(avg_col))
        dev_exprs.append((F.col(col_name) - F.avg(col_name).over(w_daily)).alias(dev_col))

    df = df.select("*", *dev_exprs)

    # Assemble features (133 total: 6 sensor + 120 lags + 6 deviation + 1 deviceId_index)
    feature_input_cols = sensor_cols + lag_cols + dev_cols + ["deviceId_index"]
    assembler = VectorAssembler(inputCols=feature_input_cols, outputCol="features")
    df_features = assembler.transform(df)

    df_features = df_features.select("properties_payloadTimeStamp", "properties_deviceID", "features")

    # --- Normalize using training data statistics (must match training scaler) ---
    train_data = spark.read.table("teams.sensorx.train_data").select("features")

    scaler = StandardScaler(inputCol="features", outputCol="features_scaled", withMean=True, withStd=True)
    scaler_model = scaler.fit(train_data)  # fit on TRAINING data, not inference data

    df_features_norm = scaler_model.transform(df_features).drop("features").withColumnRenamed("features_scaled", "features")

    return df_features_norm

In [0]:
days = 30
df_sx_data = df_read_data(days)

In [0]:
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/teams/sensorx/data-dump/mlflow_tmp"

# Load model and run predictions
model = mlflow.spark.load_model("models:/teams.sensorx.mlp_fault_prediction/5")
predictions = model.transform(df_sx_data).cache()

# Materialize now so downstream cells are instant
row_count = predictions.count()
print(f"Predictions materialized: {row_count:,} rows")

## Output

In [0]:
# Latest 24h predictions → which machines are at risk of failing within the next 15 days (our horizon)

w_rank = Window.partitionBy("properties_deviceID").orderBy(F.col("fault_probability").desc())

results = predictions.select(
    "properties_payloadTimeStamp",
    "properties_deviceID",
    F.col("prediction").cast("int").alias("predicted_label"),
    F.round(vector_to_array("probability")[1], 4).alias("fault_probability"),
)

maintenance_table = (
    results
    .filter(F.col("properties_payloadTimeStamp") >= F.current_timestamp() - F.expr("INTERVAL 24 HOURS"))
    .withColumn("rn", F.row_number().over(w_rank))
    .filter(F.col("rn") == 1)
    .drop("rn", "properties_payloadTimeStamp", "predicted_label")
    .withColumnRenamed("properties_deviceID", "machine_id")
    .withColumn("risk_level",
        F.when(F.col("fault_probability") >= 0.80, "High")
         .when(F.col("fault_probability") >= 0.50, "Medium")
         .otherwise("Low")
    )
    .withColumn("recommended_action",
        F.when(F.col("fault_probability") >= 0.80, "Inspect immediately")
         .when(F.col("fault_probability") >= 0.50, "Schedule maintenance")
         .otherwise("Continue monitoring")
    )
    .select("machine_id", "fault_probability", "risk_level", "recommended_action")
    .orderBy(F.col("fault_probability").desc())
)

display(maintenance_table)